# Phase 1 -- Deep Learning Starting Kit (PyTorch MLP)

This notebook implements **quantile regression with a neural network**.

**Prerequisites**:
- Run `1_feature_engineering.ipynb` first

**Runtime**: ~???

## 1. Setup

In [1]:
import os, sys
# Ensure we're in the notebook's directory (participant_kit/phase1/)
_this_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else None
if _this_dir is None:
    for _candidate in [os.getcwd(), os.path.join(os.getcwd(), "participant_kit", "phase1")]:
        if os.path.exists(os.path.join(_candidate, "utils.py")):
            os.chdir(_candidate)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

from utils import (
    REGIONS, HORIZONS, HOURS,
    load_train_data, load_inference_features, load_vertical_ratios,
    load_or_compute_selection,
    expand_to_all_levels, merge_speed_direction,
    winkler_score, circular_mae,
    attach_station_predictions,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)

# Device: MPS for Apple Silicon, CUDA for NVIDIA, CPU fallback
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"PyTorch device: {DEVICE}")

# -- Paths --
DATA_DIR = Path("phase1_dataset")
FEATURES_DIR = DATA_DIR / "features"

assert FEATURES_DIR.exists(), (
    f"Features directory not found: {FEATURES_DIR}\n"
    "Run 1_feature_engineering.ipynb first!"
)
print(f"Features directory: {FEATURES_DIR.resolve()}")


PyTorch device: cpu
Features directory: C:\Pythonwork\Hackathon-Sea-Winds-Predictions\phase_1\phase1_dataset\features


## 2. Load Data

We use `load_train_data()` from `utils.py` which returns the training DataFrame
along with automatically identified feature columns and target columns.

In [2]:
# Load training data for both regions
reanalysis_feat_all = {}
feature_cols_all = {}
speed_targets = None
dir_targets = None

for region in REGIONS:
    df, fcols, stgts, dtgts = load_train_data(FEATURES_DIR, region)
    reanalysis_feat_all[region] = df
    feature_cols_all[region] = fcols
    if speed_targets is None:
        speed_targets = stgts
        dir_targets = dtgts
    print(f"\n{region}:")
    print(f"  Shape: {df.shape}")
    print(f"  Features: {len(fcols)}")
    print(f"  Speed targets: {len(stgts)}, Direction targets: {len(dtgts)}")
    print(f"  Time range: {df['time'].min()} to {df['time'].max()}")

print(f"\nSpeed targets: {speed_targets}")
print(f"Direction targets: {dir_targets}")



north_sea:
  Shape: (2811240, 270)
  Features: 245
  Speed targets: 12, Direction targets: 12
  Time range: 2019-01-01 00:00:00 to 2021-12-31 00:00:00

east_china_sea:
  Shape: (2811240, 270)
  Features: 245
  Speed targets: 12, Direction targets: 12
  Time range: 2019-01-01 00:00:00 to 2021-12-31 00:00:00

Speed targets: ['speed_d14_h0', 'speed_d14_h12', 'speed_d14_h18', 'speed_d14_h6', 'speed_d1_h0', 'speed_d1_h12', 'speed_d1_h18', 'speed_d1_h6', 'speed_d7_h0', 'speed_d7_h12', 'speed_d7_h18', 'speed_d7_h6']
Direction targets: ['dir_d14_h0', 'dir_d14_h12', 'dir_d14_h18', 'dir_d14_h6', 'dir_d1_h0', 'dir_d1_h12', 'dir_d1_h18', 'dir_d1_h6', 'dir_d7_h0', 'dir_d7_h12', 'dir_d7_h18', 'dir_d7_h6']


## 3. Feature Selection

We use LGBM-based importance ranking from `utils.py` to select features per target.
DL models are more robust to noisy features (thanks to dropout), so we use a **larger
feature budget** than the GBDT starting kit:
- +1d: 20 features (vs 15 for GBDT)
- +7d: 30 features
- +14d: all features (DL can handle it)

In [3]:
# DL gets more features -- dropout handles noise better than trees
TOP_K_DL = {1: 20, 7: 30, 14: None}  # None = use all features for +14d

selected_features_all = {}
for region in REGIONS:
    cache_path = FEATURES_DIR / f"feature_selection_dl_{region}.json"
    selected = load_or_compute_selection(
        cache_path,
        reanalysis_feat_all[region],
        feature_cols_all[region],
        speed_targets,
        model_type="lgbm",
        top_k=TOP_K_DL,
    )
    selected_features_all[region] = selected
    # Show feature counts per horizon
    for h in HORIZONS:
        tgts = [t for t in speed_targets if int(t.split('_')[1][1:]) == h]
        n = np.mean([len(selected[t]) for t in tgts])
        print(f"  {region} d{h}: avg {n:.0f} features")


Computing feature selection (lgbm)...
Saved: feature_selection_dl_north_sea.json
  north_sea d1: avg 20 features
  north_sea d7: avg 30 features
  north_sea d14: avg 245 features
Computing feature selection (lgbm)...
Saved: feature_selection_dl_east_china_sea.json
  east_china_sea d1: avg 20 features
  east_china_sea d7: avg 30 features
  east_china_sea d14: avg 245 features


## 4. Train/Val Split + Scaling

**Key differences from GBDT:**
- **Feature scaling is CRITICAL.** Neural networks are sensitive to feature magnitudes.
  StandardScaler (zero mean, unit variance) ensures all features contribute equally.
- **Target normalization** per horizon: wind speeds at +1d vs +14d have different
  distributions. Normalizing targets helps the network converge faster and produce
  better-calibrated quantiles.

In [4]:
MAX_TRAIN_SAMPLES = 500_000

df_train_all = {}
df_val_all = {}

for region in REGIONS:
    df = reanalysis_feat_all[region]

    # Train: 2019-2020, Val: 2021
    # Using a full year for validation gives robust estimates across seasons
    train_mask = df["time"].dt.year.isin([2019, 2020])
    val_mask = df["time"].dt.year == 2021

    df_tr = df[train_mask].copy()
    df_vl = df[val_mask].copy()

    # Subsample training data to keep runtime ~10 min
    if len(df_tr) > MAX_TRAIN_SAMPLES:
        df_tr = df_tr.sample(n=MAX_TRAIN_SAMPLES, random_state=42)
        print(f"  {region}: subsampled to {MAX_TRAIN_SAMPLES:,} rows")

    df_train_all[region] = df_tr
    df_val_all[region] = df_vl
    print(f"{region}: train={len(df_tr):,}, val={len(df_vl):,}")


  north_sea: subsampled to 500,000 rows
north_sea: train=500,000, val=936,225
  east_china_sea: subsampled to 500,000 rows
east_china_sea: train=500,000, val=936,225


## 5. Model Architecture

A simple MLP with:
- **BatchNorm** as the first layer (alternative to external StandardScaler, but we use both for robustness)
- **SiLU** activation (smoother than ReLU, better gradients near zero)
- **Dropout** for regularization (DL's main defense against overfitting on tabular data)
- **3 outputs** per model: q05, q50, q95 (the three quantiles we need)

We train one MLP per speed target per region (24 models total). This is simpler than
a multi-output model and lets each target learn its own feature interactions.

In [5]:
class QuantileMLP(nn.Module):
    """MLP that outputs 3 quantiles (q05, q50, q95) for a single target."""

    def __init__(self, n_features, hidden=256, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(n_features),
            nn.Linear(n_features, hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 3),  # q05, q50, q95
        )

    def forward(self, x):
        return self.net(x)


def pinball_loss(pred, actual, quantiles=(0.05, 0.50, 0.95)):
    """Pinball (quantile) loss -- the proper scoring rule for quantile regression.

    pred: (batch, 3) -- predicted q05, q50, q95
    actual: (batch,) -- true values
    
    For each quantile q, penalizes under-prediction by q*|e| and over by (1-q)*|e|.
    """
    losses = []
    for i, q in enumerate(quantiles):
        errors = actual - pred[:, i]  # (batch,)
        losses.append(torch.mean(torch.max(torch.tensor(q, device=pred.device) * errors,
                                           torch.tensor(q - 1, device=pred.device) * errors)))
    return sum(losses) / len(losses)


## 6. Training Loop

We train one MLP per speed target per region.

In [6]:
# Hyperparameters
HIDDEN = 256
DROPOUT = 0.15
LR = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 2048
MAX_EPOCHS = 30
PATIENCE = 10
GRAD_CLIP = 1.0


def train_one_target(X_train, y_train, X_val, y_val, n_features, target_name):
    """Train a QuantileMLP for one speed target. Returns (model, scaler, y_mean, y_std)."""

    # --- Target normalization ---
    # Wind speed distributions differ by horizon. Normalizing helps the network
    # output values in a similar range regardless of target, improving convergence.
    y_mean = float(np.nanmean(y_train))
    y_std = float(np.nanstd(y_train))
    if y_std < 1e-6:
        y_std = 1.0  # safety for constant targets
    y_tr_norm = (y_train - y_mean) / y_std
    y_vl_norm = (y_val - y_mean) / y_std

    # --- Feature scaling ---
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train)
    X_vl_s = scaler.transform(X_val)

    # --- DataLoaders ---
    train_ds = TensorDataset(
        torch.tensor(X_tr_s, dtype=torch.float32),
        torch.tensor(y_tr_norm, dtype=torch.float32),
    )
    val_ds = TensorDataset(
        torch.tensor(X_vl_s, dtype=torch.float32),
        torch.tensor(y_vl_norm, dtype=torch.float32),
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

    # --- Model + optimizer ---
    model = QuantileMLP(n_features, hidden=HIDDEN, dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(MAX_EPOCHS):
        # --- Train ---
        model.train()
        train_loss = 0.0
        n_batches = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            loss = pinball_loss(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_loss += loss.item()
            n_batches += 1
        scheduler.step()
        train_loss /= max(n_batches, 1)

        # --- Validate ---
        model.eval()
        val_loss = 0.0
        n_val = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb)
                loss = pinball_loss(pred, yb)
                val_loss += loss.item()
                n_val += 1
        val_loss /= max(n_val, 1)

        # --- Early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

    return model, scaler, y_mean, y_std


In [7]:
def predict_with_mlp(model, scaler, X_raw, y_mean, y_std):
    """Run inference with a trained MLP. Returns (q05, q50, q95) arrays.

    Steps: scale features -> forward pass -> denormalize -> enforce non-negative.
    """
    model.eval()
    X_s = scaler.transform(X_raw)
    with torch.no_grad():
        preds = model(
            torch.tensor(X_s, dtype=torch.float32).to(DEVICE)
        ).cpu().numpy()

    # Denormalize: reverse the (y - mean) / std transformation
    q05 = np.maximum(preds[:, 0] * y_std + y_mean, 0)
    q50 = preds[:, 1] * y_std + y_mean
    q95 = preds[:, 2] * y_std + y_mean

    # Enforce quantile ordering: q05 <= q50 <= q95
    stacked = np.column_stack([q05, q50, q95])
    stacked.sort(axis=1)
    return stacked[:, 0], stacked[:, 1], stacked[:, 2]

In [8]:
# Train all speed targets for both regions
# This is the main training loop -- takes ~8 minutes

dl_models = {}      # {region: {target: (model, scaler, y_mean, y_std)}}
dl_results = {}     # {region: DataFrame}

for region in REGIONS:
    print(f"\n{'='*60}")
    print(f"Training DL models for {region}")
    print(f"{'='*60}")

    df_tr = df_train_all[region]
    df_vl = df_val_all[region]
    models_r = {}
    results_r = []

    for tgt in speed_targets:
        horizon = int(tgt.split("_")[1][1:])
        hour = int(tgt.split("_")[2][1:])

        # Get selected features for this target
        feats = selected_features_all[region].get(tgt, feature_cols_all[region])

        # Filter valid rows (non-NaN target)
        mask_tr = df_tr[tgt].notna()
        mask_vl = df_vl[tgt].notna()
        if mask_tr.sum() < 100 or mask_vl.sum() < 100:
            print(f"  {tgt}: skipped (insufficient data)")
            continue

        X_tr = df_tr.loc[mask_tr, feats].fillna(0).values
        y_tr = df_tr.loc[mask_tr, tgt].values
        X_vl = df_vl.loc[mask_vl, feats].fillna(0).values
        y_vl = df_vl.loc[mask_vl, tgt].values

        # Train
        model, scaler, y_mean, y_std = train_one_target(
            X_tr, y_tr, X_vl, y_vl, len(feats), tgt
        )
        models_r[tgt] = (model, scaler, y_mean, y_std)

        # Evaluate on validation set
        q05, q50, q95 = predict_with_mlp(model, scaler, X_vl, y_mean, y_std)
        ws = winkler_score(y_vl, q05, q95)
        rmse = float(np.sqrt(np.nanmean((y_vl - q50) ** 2)))
        cov = float(np.mean((y_vl >= q05) & (y_vl <= q95)) * 100)

        results_r.append({
            "target": tgt, "horizon": horizon, "hour": hour,
            "winkler": ws, "rmse": rmse, "coverage": cov,
            "n_features": len(feats),
        })
        print(f"  {tgt}: WS={ws:.2f}, RMSE={rmse:.2f}, cov={cov:.0f}% ({len(feats)} feats)")

    dl_models[region] = models_r
    dl_results[region] = pd.DataFrame(results_r)

    print(f"\n{region} -- Mean by horizon:")
    print(dl_results[region].groupby("horizon")[["winkler", "rmse", "coverage"]].mean().round(2))


Training DL models for north_sea
  speed_d14_h0: WS=12.82, RMSE=3.16, cov=82% (245 feats)
  speed_d14_h12: WS=13.32, RMSE=3.22, cov=80% (245 feats)
  speed_d14_h18: WS=12.97, RMSE=3.18, cov=82% (245 feats)
  speed_d14_h6: WS=13.06, RMSE=3.17, cov=80% (245 feats)
  speed_d1_h0: WS=3.71, RMSE=0.92, cov=90% (20 feats)
  speed_d1_h12: WS=4.67, RMSE=1.13, cov=90% (20 feats)
  speed_d1_h18: WS=5.04, RMSE=1.20, cov=90% (20 feats)
  speed_d1_h6: WS=4.07, RMSE=1.00, cov=90% (20 feats)
  speed_d7_h0: WS=10.88, RMSE=2.86, cov=88% (30 feats)
  speed_d7_h12: WS=12.04, RMSE=3.02, cov=86% (30 feats)
  speed_d7_h18: WS=11.32, RMSE=2.95, cov=87% (30 feats)
  speed_d7_h6: WS=11.08, RMSE=2.89, cov=88% (30 feats)

north_sea -- Mean by horizon:
         winkler  rmse  coverage
horizon                         
1           4.37  1.06     89.87
7          11.33  2.93     87.24
14         13.04  3.18     80.88

Training DL models for east_china_sea
  speed_d14_h0: WS=12.29, RMSE=2.95, cov=82% (245 feats)
  sp

## 7. Direction Models (LGBM sin/cos)

In [9]:
# Direction Models — LGBM sin/cos
# Self-contained: loads data directly, no dependency on other cells' variables
import lightgbm as lgb
from utils import load_train_data, exclude_worldwide_features, circular_mae
import time as _time

dir_models = {}
t0 = _time.time()

for region in REGIONS:
    df_dir, feature_cols_dir, _, dir_targets_list = load_train_data(FEATURES_DIR, region)
    df_train_dir = df_dir[df_dir["time"].dt.year.isin([2019, 2020])]
    
    dir_models[region] = {}
    for tgt in dir_targets_list:
        mask_tr = df_train_dir[tgt].notna()
        if mask_tr.sum() < 100:
            continue
        
        # Feature selection (top-25)
        X_sub = df_train_dir.loc[mask_tr, feature_cols_dir].fillna(0).sample(
            n=min(100_000, mask_tr.sum()), random_state=42)
        y_sub = df_train_dir.loc[X_sub.index, tgt]
        m_imp = lgb.LGBMRegressor(n_estimators=50, max_depth=4, verbose=-1, n_jobs=-1)
        m_imp.fit(X_sub, np.sin(np.radians(y_sub)))
        dir_feats = pd.Series(m_imp.feature_importances_, index=feature_cols_dir).nlargest(25).index.tolist()
        
        X_tr = df_train_dir.loc[mask_tr, dir_feats].fillna(0)
        y_tr_sin = np.sin(np.radians(df_train_dir.loc[mask_tr, tgt]))
        y_tr_cos = np.cos(np.radians(df_train_dir.loc[mask_tr, tgt]))
        
        params = dict(n_estimators=200, max_depth=7, learning_rate=0.05,
                      subsample=0.8, verbose=-1, n_jobs=-1)
        m_sin = lgb.LGBMRegressor(**params)
        m_cos = lgb.LGBMRegressor(**params)
        m_sin.fit(X_tr, y_tr_sin)
        m_cos.fit(X_tr, y_tr_cos)
        dir_models[region][tgt] = (m_sin, m_cos, dir_feats)
    
    print(f"  {region}: {len(dir_models[region])} direction models trained")
    del df_dir, df_train_dir

print(f"Direction training complete in {_time.time() - t0:.0f}s")


  north_sea: 12 direction models trained
  east_china_sea: 12 direction models trained
Direction training complete in 474s


In [10]:
dir_offsets = {}  # fallback if computation fails
try:
    # Compute direction prediction intervals
    from utils import compute_direction_intervals
    
    dir_offsets = {}
    for region in REGIONS:
        df_tr_dir, fcols_dir, _, dir_tgts = load_train_data(FEATURES_DIR, region)
        df_tr_dir = df_tr_dir[df_tr_dir["time"].dt.year.isin([2019, 2020])]
        dir_offsets[region] = compute_direction_intervals(
            df_tr_dir, dir_tgts, fcols_dir, dir_models[region], quantile_hi=0.975)
        print(f"  {region}: " + ", ".join(f"+{h}d: +/-{v:.1f} deg" for h, v in dir_offsets[region].items()))
        del df_tr_dir
    
except Exception as e:
    print(f"Direction interval computation failed: {e}")
    print("Submission will not have dir_05/dir_95 columns")


  north_sea: +1d: +/-62.6 deg, +7d: +/-138.2 deg, +14d: +/-136.1 deg
  east_china_sea: +1d: +/-87.4 deg, +7d: +/-142.8 deg, +14d: +/-144.8 deg


## 8. Evaluation Summary

In [11]:
try:
    for region in REGIONS:
        print(f"\n{'='*60}")
        print(f"DL Results: {region}")
        print(f"{'='*60}")
    
        df_r = dl_results[region]
        if df_r.empty:
            print("  No results")
            continue
    
        print(f"\n{'Horizon':<10} {'Winkler':>8} {'RMSE':>8} {'Coverage':>9}")
        print("-" * 40)
        for h in HORIZONS:
            sub = df_r[df_r["horizon"] == h]
            if sub.empty:
                continue
            print(f"  +{h:>2}d     {sub['winkler'].mean():>8.2f} {sub['rmse'].mean():>8.2f} "
                  f"{sub['coverage'].mean():>8.1f}%")
        print(f"  {'Overall':<8} {df_r['winkler'].mean():>8.2f} {df_r['rmse'].mean():>8.2f} "
              f"{df_r['coverage'].mean():>8.1f}%")
    
        # Direction
        dir_r = {}
        if not dir_r.empty:
            print(f"\nDirection cMAE by horizon:")
            print(dir_r.groupby("horizon")["cmae"].mean().round(1).to_string())
    
except Exception as e:
    print(f"Evaluation summary skipped: {e}")



DL Results: north_sea

Horizon     Winkler     RMSE  Coverage
----------------------------------------
  + 1d         4.37     1.06     89.9%
  + 7d        11.33     2.93     87.2%
  +14d        13.04     3.18     80.9%
  Overall      9.58     2.39     86.0%
Evaluation summary skipped: 'dict' object has no attribute 'empty'


## 9. Generate Submission

For DL inference, we need to:
1. Load pre-computed inference features for each window/region
2. Scale features with the **same scaler** used during training
3. Forward pass through the MLP
4. **Denormalize** predictions (reverse the target normalization)
5. Add direction predictions
6. Expand 10m predictions to all 7 vertical levels

We write a custom inference loop because the DL models need the scaler and
normalization parameters, unlike GBDT models which work on raw features.

In [12]:
# Load vertical ratios for level expansion
vertical_ratios = {}
for region in REGIONS:
    vertical_ratios[region] = load_vertical_ratios(FEATURES_DIR, region)
    has_ratios = vertical_ratios[region] is not None
    print(f"{region}: vertical ratios {'loaded' if has_ratios else 'not found (using power-law fallback)'}")

north_sea: vertical ratios loaded
east_china_sea: vertical ratios loaded


In [13]:
from utils import add_direction_intervals
NUM_WINDOWS = 8
all_predictions = []

for wid in range(1, NUM_WINDOWS + 1):
    for region in REGIONS:
        df_inf = load_inference_features(FEATURES_DIR, wid, region)
        context_month = int(df_inf["time"].max().month)
        fcols = feature_cols_all[region]

        # Ensure all feature columns exist in inference data
        for c in fcols:
            if c not in df_inf.columns:
                df_inf[c] = 0.0

        # --- Speed predictions at 10m ---
        rows = []
        for tgt, (model, scaler, y_mean, y_std) in dl_models[region].items():
            horizon = int(tgt.split("_")[1][1:])
            hour = int(tgt.split("_")[2][1:])
            feats = selected_features_all[region].get(tgt, fcols)

            X_inf = df_inf[feats].fillna(0).values
            q05, q50, q95 = predict_with_mlp(model, scaler, X_inf, y_mean, y_std)

            for j in range(len(df_inf)):
                rows.append({
                    "latitude": round(float(df_inf.iloc[j]["latitude"]), 2),
                    "longitude": round(float(df_inf.iloc[j]["longitude"]), 2),
                    "horizon": horizon,
                    "hour": hour,
                    "q05": round(float(q05[j]), 4),
                    "q50": round(float(q50[j]), 4),
                    "q95": round(float(q95[j]), 4),
                })

        preds_10m = pd.DataFrame(rows)

        # --- Direction predictions ---
        dir_preds = {}
        for tgt, (m_sin, m_cos, dir_feats) in dir_models[region].items():
            horizon = int(tgt.split("_")[1][1:])
            hour = int(tgt.split("_")[2][1:])
            for c in dir_feats:
                if c not in df_inf.columns:
                    df_inf[c] = 0.0
            X_dir = df_inf[dir_feats].fillna(0)
            dp = np.degrees(np.arctan2(
                m_sin.predict(X_dir), m_cos.predict(X_dir)
            )) % 360
            dir_preds[(horizon, hour)] = dp

        preds_10m = merge_speed_direction(preds_10m, dir_preds)
        # Direction intervals are REQUIRED by the scorer (circular Winkler on
        # the 90% PI). Use calibrated offsets if available, else fall back to
        # a ±90° symmetric arc so the submission still validates.
        if dir_offsets and region in dir_offsets:
            preds_10m = add_direction_intervals(preds_10m, dir_offsets[region])
        else:
            preds_10m = add_direction_intervals(
                preds_10m, {h: 90.0 for h in (1, 7, 14)}
            )

        # --- Expand to all 7 vertical levels ---
        preds = expand_to_all_levels(
            preds_10m, vertical_ratios.get(region), context_month
        )
        preds["window"] = wid
        preds["region"] = region
        all_predictions.append(preds)

        n_levels = preds["level"].nunique()
        print(f"  W{wid} {region}: {len(preds):,} predictions ({n_levels} levels)")


  W1 north_sea: 215,460 predictions (7 levels)
  W1 east_china_sea: 215,460 predictions (7 levels)
  W2 north_sea: 215,460 predictions (7 levels)
  W2 east_china_sea: 215,460 predictions (7 levels)
  W3 north_sea: 215,460 predictions (7 levels)
  W3 east_china_sea: 215,460 predictions (7 levels)
  W4 north_sea: 215,460 predictions (7 levels)
  W4 east_china_sea: 215,460 predictions (7 levels)
  W5 north_sea: 215,460 predictions (7 levels)
  W5 east_china_sea: 215,460 predictions (7 levels)
  W6 north_sea: 215,460 predictions (7 levels)
  W6 east_china_sea: 215,460 predictions (7 levels)
  W7 north_sea: 215,460 predictions (7 levels)
  W7 east_china_sea: 215,460 predictions (7 levels)
  W8 north_sea: 215,460 predictions (7 levels)
  W8 east_china_sea: 215,460 predictions (7 levels)


In [14]:
# Assemble and save submission
if all_predictions:
    submission = pd.concat(all_predictions, ignore_index=True)
    out_cols = ["window", "region", "latitude", "longitude",
               "horizon", "hour", "level", "q05", "q50", "q95", "dir_50"]
    if "dir_05" in submission.columns:
        out_cols += ["dir_05", "dir_95"]
    submission = submission[out_cols]

    # Enforce physical constraints
    submission["q05"] = submission["q05"].clip(lower=0)
    submission["q95"] = submission[["q50", "q95"]].max(axis=1)
    submission["q05"] = submission[["q05", "q50"]].min(axis=1)
    submission["dir_50"] = submission["dir_50"] % 360

    # Attach station predictions (baseline: inherit from nearest grid point)
    submission = attach_station_predictions(submission, FEATURES_DIR)

    out_path = "predictions_dl.csv"
    submission.to_csv(out_path, index=False)
    print(f"\nSubmission saved: {out_path}")
    # --- Package for Codabench upload ---
    # Codabench expects a ZIP containing a file named exactly "predictions.csv".
    import zipfile
    submission_zip = Path("submission_dl.zip")
    with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(Path(out_path), arcname="predictions.csv")
    print(f"Ready to upload to Codabench: {submission_zip}")

    print(f"Shape: {submission.shape}")
    print(f"Windows: {sorted(submission['window'].unique())}")
    print(f"Regions: {sorted(submission['region'].unique())}")
    print(f"Levels: {sorted(submission['level'].unique())}")
    print(f"\nSample:")
    print(submission.head(10))
else:
    print("No predictions generated!")



Submission saved: predictions_dl.csv
Ready to upload to Codabench: submission_dl.zip
Shape: (3448800, 15)
Windows: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
Regions: ['east_china_sea', 'north_sea']
Levels: ['', '1000', '100m', '10m', '500', '700', '850', '925']

Sample:
   type  window     region  latitude  longitude station  horizon  hour level  \
0  grid       1  north_sea      51.0      -4.00               14     0   10m   
1  grid       1  north_sea      51.0      -3.75               14     0   10m   
2  grid       1  north_sea      51.0      -3.50               14     0   10m   
3  grid       1  north_sea      51.0      -3.25               14     0   10m   
4  grid       1  north_sea      51.0      -3.00               14     0   10m   
5  grid       1  north_sea      51.0      -2.75               14     0   10m   
6  grid       1  north_sea      51.0      -2.50               14     0   10m   
7  grid       1  north_se